# Banco de dados
![Diagrama ER](../banco/diagrama.png)

In [35]:
import json
import sqlite3
import pandas as pd
from pathlib import Path
from utils import carregar_dados

dados = carregar_dados()

# amenities: lista dentro de um campo -> tabela própria + tabela de ligação (N:N)
lista = dados["hotels"]["amenities"].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
pares = dados["hotels"][["hotel_id"]].assign(amenity=lista).explode("amenity").dropna()
amenities = pd.DataFrame({"amenity": sorted(pares["amenity"].unique())})
amenities.insert(0, "amenity_id", range(1, len(amenities) + 1))
dados["amenities"] = amenities
dados["hotel_amenities"] = pares.merge(amenities, on="amenity")[["hotel_id", "amenity_id"]]

# remover colunas repetidas ou derivadas
dados["daily_market"] = dados["daily_market"].drop(columns=["hotel_name", "tier", "comp_set", "available_rooms"])
dados["weekly_performance"] = dados["weekly_performance"].drop(columns=["hotel_name", "tier", "comp_set"])
dados["room_types"] = dados["room_types"].drop(columns=["tier"])
dados["hotels"] = dados["hotels"].drop(columns=["amenities", "review_count", "lifetime_rating",
                                                "share_rating_5", "share_rating_1_2"])

# datas como texto AAAA-MM-DD, para as chaves das tabelas baterem
datas = {"daily_market": ["date"], "weekly_performance": ["week_start"],
         "rate_decisions": ["week_start"], "reviews": ["review_date", "stay_date", "week_start"]}
for tabela, colunas in datas.items():
    for c in colunas:
        dados[tabela][c] = pd.to_datetime(dados[tabela][c]).dt.strftime("%Y-%m-%d")

print({nome: df.shape for nome, df in dados.items()})

{'hotels': (14, 12), 'room_types': (42, 9), 'daily_market': (10192, 52), 'weekly_performance': (1456, 38), 'rate_decisions': (1442, 24), 'reviews': (12077, 26), 'review_scenarios': (11, 7), 'amenities': (33, 2), 'hotel_amenities': (57, 2)}


In [36]:
TIPOS = {"int64": "INTEGER", "float64": "REAL", "bool": "INTEGER"}

def criar_tabela(con, nome, df, pk, regras=()):
    colunas = [f'"{c}" {TIPOS.get(str(t), "TEXT")}' for c, t in df.dtypes.items()]
    todas = colunas + [f"PRIMARY KEY ({pk})", *regras]
    con.execute(f"CREATE TABLE {nome} (\n  " + ",\n  ".join(todas) + "\n)")

FK_HOTEL = "FOREIGN KEY (hotel_id) REFERENCES hotels(hotel_id)"
FK_SEMANA = "FOREIGN KEY (hotel_id, week_start) REFERENCES weekly_performance(hotel_id, week_start)"
FK_QUARTO = "FOREIGN KEY (hotel_id, room_type) REFERENCES room_types(hotel_id, room_type)"

In [37]:
quartos = dados["room_types"][["hotel_id", "room_type"]]
teste = dados["reviews"][["hotel_id", "room_type"]].drop_duplicates().merge(quartos, how="left", indicator=True)
print((teste["_merge"] == "left_only").sum())

0


In [38]:
if "con" in globals():
    con.close()

caminho = Path("../banco/nice_rooms.db")
caminho.parent.mkdir(exist_ok=True)
caminho.unlink(missing_ok=True)

con = sqlite3.connect(caminho)
con.execute("PRAGMA foreign_keys = ON")

criar_tabela(con, "hotels", dados["hotels"], "hotel_id",
             ["CHECK (tier IN ('luxury','upscale','boutique','midscale','economy'))"])
criar_tabela(con, "amenities", dados["amenities"], "amenity_id", ["UNIQUE (amenity)"])
criar_tabela(con, "hotel_amenities", dados["hotel_amenities"], "hotel_id, amenity_id",
             [FK_HOTEL, "FOREIGN KEY (amenity_id) REFERENCES amenities(amenity_id)"])
criar_tabela(con, "review_scenarios", dados["review_scenarios"], "scenario_id")
criar_tabela(con, "room_types", dados["room_types"], "room_type_id",
             [FK_HOTEL, "UNIQUE (hotel_id, room_type)"])
criar_tabela(con, "weekly_performance", dados["weekly_performance"], "hotel_id, week_start",
             [FK_HOTEL, "CHECK (occupancy BETWEEN 0 AND 1)"])
criar_tabela(con, "daily_market", dados["daily_market"], "hotel_id, date",
             [FK_HOTEL, "CHECK (occupancy BETWEEN 0 AND 1)"])
criar_tabela(con, "rate_decisions", dados["rate_decisions"], "decision_id",
             [FK_HOTEL, FK_SEMANA, "UNIQUE (hotel_id, week_start)",
              "CHECK (action IN ('raise','hold','lower'))"])
criar_tabela(con, "reviews", dados["reviews"], "review_id",
             [FK_HOTEL, FK_SEMANA, FK_QUARTO,
              "FOREIGN KEY (scenario_id) REFERENCES review_scenarios(scenario_id)",
              "CHECK (rating BETWEEN 1 AND 5)"])


In [39]:
ORDEM = ["hotels", "amenities", "hotel_amenities", "review_scenarios", "room_types",
         "weekly_performance", "daily_market", "rate_decisions", "reviews"]

for nome in ORDEM:
    dados[nome].to_sql(nome, con, if_exists="append", index=False)
con.commit()
print("Dados carregados")

Dados carregados


In [40]:
for nome in ORDEM:
    print(nome, con.execute(f"SELECT COUNT(*) FROM {nome}").fetchone()[0])

print("Referências quebradas:", con.execute("PRAGMA foreign_key_check").fetchall())

hotels 14
amenities 33
hotel_amenities 57
review_scenarios 11
room_types 42
weekly_performance 1456
daily_market 10192
rate_decisions 1442
reviews 12077
Referências quebradas: []


In [41]:
con.execute("DROP VIEW IF EXISTS v_semanal")
con.execute("""CREATE VIEW v_semanal AS
SELECT w.*, h.name AS hotel_name, h.tier, h.comp_set,
       CAST(substr(w.week_start, 1, 4) AS INTEGER) AS ano
FROM weekly_performance w JOIN hotels h USING (hotel_id)""")
con.commit()
pd.read_sql("SELECT * FROM v_semanal LIMIT 3", con)

,hotel_id,week_start,available_rooms,rooms_sold,bookings,cancellations,occupancy,adr,revpar,room_revenue,...,flow_through,review_count,mean_rating,positive_share,sentiment_index,response_rate,hotel_name,tier,comp_set,ano
0,H01,2027-01-04,1176,732,871,139,0.6224,187.76,116.87,137440.25,...,None,8,4.250,0.7500,0.2365,0.5000,Alder & Quay Hotel,upscale,upscale,2027
1,H01,2027-01-11,1176,738,879,141,0.6276,189.37,118.84,139754.61,...,None,9,3.889,0.5556,0.2452,0.5556,Alder & Quay Hotel,upscale,upscale,2027
2,H01,2027-01-18,1176,741,881,140,0.6301,188.52,118.79,139695.80,...,None,5,2.600,0.0000,-0.4753,1.0000,Alder & Quay Hotel,upscale,upscale,2027


In [42]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../banco/nice_rooms.db")

def sql(consulta):
    return pd.read_sql(consulta, con)

In [43]:
sql("""SELECT ROUND(SUM(room_revenue)) AS receita_quartos,
              ROUND(SUM(room_revenue + ancillary_revenue)) AS receita_total
       FROM daily_market""")

,receita_quartos,receita_total
0,202759645.0,236002362.0


In [44]:
sql("""
SELECT d.year AS ano, h.name AS hotel, h.tier,
       ROUND(SUM(d.room_revenue + d.ancillary_revenue)) AS receita_total,
       ROUND(SUM(d.room_revenue + d.ancillary_revenue) / COUNT(*)) AS receita_por_noite,
       RANK() OVER (PARTITION BY d.year
                    ORDER BY SUM(d.room_revenue + d.ancillary_revenue) DESC) AS posicao
FROM daily_market d
JOIN hotels h USING (hotel_id)
GROUP BY d.year, h.hotel_id
ORDER BY ano, posicao
""")

,ano,hotel,tier,receita_total,receita_por_noite,posicao
0,2027,Marisol Bay Resort,luxury,26118854.0,72152.0,1
1,2027,Seabright Grand,upscale,16138180.0,44581.0,2
2,2027,Alder & Quay Hotel,upscale,12297826.0,33972.0,3
3,2027,Harborlight Suites,upscale,10994515.0,30372.0,4
4,2027,The Copper Meridian,luxury,10609145.0,29307.0,5
5,2027,Kestrel Park Hotel,midscale,8236615.0,22753.0,6
6,2027,Willowmere Lodge,upscale,6515026.0,17997.0,7
7,2027,Cedar & Salt Hotel,midscale,6238101.0,17232.0,8
8,2027,Paloma Court Hotel,midscale,5761487.0,15916.0,9
9,2027,The Lantern House,boutique,4608801.0,12731.0,10


In [45]:
sql("""
SELECT h.tier, d.season AS estacao,
       ROUND(100.0 * SUM(d.rooms_sold) / SUM(d.rooms_available), 1) AS ocupacao_pct,
       ROUND(SUM(d.room_revenue) / SUM(d.rooms_sold), 2) AS adr
FROM daily_market d
JOIN hotels h USING (hotel_id)
GROUP BY h.tier, d.season
ORDER BY CASE h.tier WHEN 'luxury' THEN 1 WHEN 'upscale' THEN 2 WHEN 'boutique' THEN 3
                     WHEN 'midscale' THEN 4 ELSE 5 END,
         CASE d.season WHEN 'winter' THEN 1 WHEN 'spring' THEN 2 WHEN 'summer' THEN 3 ELSE 4 END
""")

,tier,estacao,ocupacao_pct,adr
0,luxury,winter,63.1,322.20
1,luxury,spring,71.6,366.02
2,luxury,summer,83.3,454.65
3,luxury,autumn,71.6,369.96
4,upscale,winter,63.7,192.56
5,upscale,spring,72.4,218.64
6,upscale,summer,84.1,271.79
7,upscale,autumn,72.2,220.75
8,boutique,winter,66.1,185.38
9,boutique,spring,74.9,210.22


In [46]:
sql("""
SELECT h.name AS hotel, h.tier, COUNT(*) AS avaliacoes,
       SUM(r.rating <= 2) AS notas_1_2,
       ROUND(100.0 * SUM(r.rating <= 2) / COUNT(*), 1) AS pct_1_2
FROM reviews r
JOIN hotels h USING (hotel_id)
GROUP BY h.hotel_id
HAVING 100.0 * SUM(r.rating <= 2) / COUNT(*) > 30
ORDER BY pct_1_2 DESC
""")

,hotel,tier,avaliacoes,notas_1_2,pct_1_2
0,Northgate Inn,economy,718,281,39.1
1,Ridgeline Motor Inn,economy,506,184,36.4
2,The Wren Rooms,boutique,289,103,35.6
3,Kestrel Park Hotel,midscale,1169,411,35.2
4,Willowmere Lodge,upscale,679,221,32.5
5,Cedar & Salt Hotel,midscale,963,304,31.6
6,Paloma Court Hotel,midscale,829,251,30.3


In [47]:
sql("""
SELECT s.label AS cenario, s.rating_band AS faixa_de_notas, COUNT(*) AS avaliacoes,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM reviews), 1) AS pct_do_total,
       ROUND(AVG(r.rating), 2) AS nota_media,
       s.recovery_action AS acao_de_recuperacao
FROM reviews r
JOIN review_scenarios s USING (scenario_id)
GROUP BY s.scenario_id
ORDER BY avaliacoes DESC
""")

,cenario,faixa_de_notas,avaliacoes,pct_do_total,nota_media,acao_de_recuperacao
0,Stay exceeded expectations,"5, 4",1840,15.2,4.84,thank the guest and record the staff member
1,Room condition,"3, 2, 1",1401,11.6,2.48,block the room and raise a work order
2,Noise and sleep,"3, 2, 1",1250,10.4,2.42,"offer a move and a sleep kit, note the room"
3,Location and access,"5, 4",1219,10.1,4.58,none needed
4,Service recovery,"5, 4, 3",1206,10.0,4.33,log the recovery and brief the shift
5,Billing and charges,"3, 2, 1",1027,8.5,1.63,audit the folio and refund where due
6,Breakfast and food,"5, 4, 3",1023,8.5,3.73,invite the guest back and review the buffet line
7,Check-in and queue,"4, 3, 2",1006,8.3,2.81,add a queue check at peak arrival
8,Cleanliness,"3, 2, 1",868,7.2,1.69,"reclean, inspect and coach the section"
9,Wifi and connectivity,"4, 3, 2",765,6.3,2.74,send the access point log to IT


In [48]:
sql("""
SELECT action AS acao, COUNT(*) AS decisoes,
       ROUND(100.0 * SUM(outcome = 'better') / COUNT(*), 1) AS pct_melhor,
       ROUND(100.0 * SUM(outcome = 'in line') / COUNT(*), 1) AS pct_igual,
       ROUND(100.0 * SUM(outcome = 'worse') / COUNT(*), 1) AS pct_pior,
       ROUND(AVG(realised_uplift_pct), 2) AS uplift_medio_pct
FROM rate_decisions
GROUP BY action
ORDER BY CASE action WHEN 'raise' THEN 1 WHEN 'hold' THEN 2 ELSE 3 END
""")

,acao,decisoes,pct_melhor,pct_igual,pct_pior,uplift_medio_pct
0,raise,634,38.3,30.8,30.9,0.45
1,hold,619,37.6,31.8,30.5,0.68
2,lower,189,42.9,26.5,30.7,0.60


In [49]:
sql("""
SELECT rating AS nota, COUNT(*) AS avaliacoes,
       SUM(COALESCE(response_text, '') <> '') AS respondidas,
       ROUND(100.0 * SUM(COALESCE(response_text, '') <> '') / COUNT(*), 1) AS taxa_resposta_pct,
       ROUND(AVG(response_hours), 1) AS horas_medias
FROM reviews
GROUP BY rating
ORDER BY rating
""")

,nota,avaliacoes,respondidas,taxa_resposta_pct,horas_medias
0,1,1087,1087,100.0,21.2
1,2,2439,2439,100.0,21.2
2,3,3212,3212,100.0,21.1
3,4,2302,663,28.8,20.2
4,5,3037,907,29.9,21.4


# Minha pergunta: Quais hotéis geram mais receita, com ou sem piscina?

In [50]:
sql("""
SELECT a.amenity AS comodidade, COUNT(*) AS hoteis
FROM amenities a
JOIN hotel_amenities USING (amenity_id)
GROUP BY a.amenity_id
ORDER BY hoteis DESC
""")

,comodidade,hoteis
0,parking,6
1,restaurant,4
2,gym,4
3,breakfast,4
4,spa,3
5,pool,3
6,design rooms,3
7,wifi,2
8,pet friendly,2
9,laundry,2


In [51]:
PISCINA = "pool"

con.execute("DROP VIEW IF EXISTS v_piscina")
con.execute(f"""
CREATE VIEW v_piscina AS
SELECT h.hotel_id, h.name AS hotel, h.tier,
       CASE WHEN EXISTS (SELECT 1
                         FROM hotel_amenities ha
                         JOIN amenities a USING (amenity_id)
                         WHERE ha.hotel_id = h.hotel_id AND a.amenity LIKE '%{PISCINA}%')
            THEN 'com piscina' ELSE 'sem piscina' END AS grupo
FROM hotels h
""")
con.commit()

In [52]:
sql("""
SELECT tier, grupo, COUNT(*) AS hoteis, GROUP_CONCAT(hotel, ', ') AS lista
FROM v_piscina
GROUP BY tier, grupo
ORDER BY CASE tier WHEN 'luxury' THEN 1 WHEN 'upscale' THEN 2 WHEN 'boutique' THEN 3
                   WHEN 'midscale' THEN 4 ELSE 5 END, grupo
""")

,tier,grupo,hoteis,lista
0,luxury,com piscina,1,Marisol Bay Resort
1,luxury,sem piscina,1,The Copper Meridian
2,upscale,com piscina,1,Seabright Grand
3,upscale,sem piscina,3,"Alder & Quay Hotel, Harborlight Suites, Willow..."
4,boutique,sem piscina,3,"The Lantern House, The Wren Rooms, The Foxglove"
5,midscale,com piscina,1,Kestrel Park Hotel
6,midscale,sem piscina,2,"Cedar & Salt Hotel, Paloma Court Hotel"
7,economy,sem piscina,2,"Northgate Inn, Ridgeline Motor Inn"


In [33]:
sql("""
SELECT p.grupo,
       COUNT(DISTINCT d.hotel_id) AS hoteis,
       ROUND(SUM(d.room_revenue + d.ancillary_revenue)) AS receita_total,
       ROUND(SUM(d.room_revenue + d.ancillary_revenue) / COUNT(DISTINCT d.hotel_id)) AS receita_media_por_hotel,
       ROUND(SUM(d.room_revenue + d.ancillary_revenue) / COUNT(*)) AS receita_media_por_hotel_noite,
       ROUND(SUM(d.room_revenue + d.ancillary_revenue) / SUM(d.rooms_available), 2) AS receita_por_quarto_noite
FROM daily_market d
JOIN v_piscina p USING (hotel_id)
GROUP BY p.grupo
ORDER BY receita_media_por_hotel DESC
""")

,grupo,hoteis,receita_total,receita_media_por_hotel,receita_media_por_hotel_noite,receita_por_quarto_noite
0,com piscina,3,101399415.0,33799805.0,46428.0,221.79
1,sem piscina,11,134602947.0,12236632.0,16809.0,163.33
